## Developing LLM Applications with Langchain

### The Langchain Ecosystem

LangChain
- LangGraph (AI Agents)
- LangSmith (Deploying Apps)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="",
    api_key=""
)

llm.invoke("Test prompt")

In [ ]:
# prompting hugging face models
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id = "the_model",
    task = "",
    pipeline_kwargs = {"max_new_tokens":100}
)


### Prompt Templates
- Defining prompts for LLMs

In [ ]:
from langchain_core.prompt import PromptTemplate

template = "What is a good name for a company that makes {product}?"

prompt_template = PromptTemplate.from_template(
    template=template
)

prompt = prompt_template.invoke({"product": "colorful socks"})
print(prompt)

In [ ]:
llm = HuggingFacePipeline.from_model_id(
    model_id = "the_model",
    task = "",
    pipeline_kwargs = {"max_new_tokens":100}
)
llm_langchain = prompt_template | llm # the "|" operator is used to chain the prompt template and the LLM together

product = "colorful socks"
print(llm_langchain.invoke({"product": product}))

### Chat models

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
        ("human", "{text}")
    ]
) # {} indicate inputs variables that will be provided when the prompt is invoked

In [ ]:
llm = ChatOpenAI(
    model="",
    api_key=""
    )

llm_chain = template | llm

response = llm_chain.invoke({
    "input_language": "English",
    "output_language": "French",
    "text": "Hello, how are you?"
})

print(response.content)

### Few-shot Prompting

In [ ]:
examples = [
    {
        "question":"",
        "answer":""
    }
]

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "Question: {question}\n{answer}"   
)

prompt = example_prompt.invoke(
    {
        "question": "What is the capital of Italy?",
        "answer" : "Rome"
    }
)

In [ ]:
prompt_template = FewShotPromptTemplate(
    examples=examples, # list of examples to include in the prompt
    example_prompt=example_prompt, # prompt template to use for each example
    suffix="Question: {input}", # suffix to use for the final input
    input_variables=["input"]
)

prompt = prompt_template.invoke(
    {
        "input": "What is the capital of France?"
    }
)

In [ ]:
# integrate with a chain

llm = ChatOpenAI(
    model="",
    api_key=""
    )

llm_chain = prompt_template | llm
response = llm_chain.invoke(
    {
        "input": "What is the capital of Germany?"
    }
)

## Sequential Chains

In [ ]:
# Output -> Input

destination_prompt = PromptTemplate(
    input_variables=["destination"],
    template="What is the best way to travel to {destination}?"
)

activity_prompt = PromptTemplate(
    input_variables=["activity"],
    template="What is the best way to {activity}?"
)

llm = ChatOpenAI(
    model="", 
    api_key=""
    )

seq_chain = (
    {
        "activities":destination_prompt | llm | StrOutputParser()
    }
    | activity_prompt 
    | llm
    | StrOutputParser()
)

In [ ]:
print(seq_chain.invoke({"destination": "Paris"}))

### Introduction to LangChain Agents

![alt text](<Screenshot 2026-10-03 111210.png>)

ReAct Agents
- Reason + Act

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_community.agent_toolkits.load_tools import load_tools

llm = ChatOpenAI(
    model="",
    api_key=""
)

tools = load_tools(["serpapi", "llm-math"], llm=llm)
agent = create_react_agent(llm=llm, tools=tools)

messages = agent.invoke({"messages" : [{"human": "What is the capital of France?"}]})

### Custom Tools for Agents

In [ ]:
from langchain_core.tools import tool

@tool
def financial_report(company_name: str, revenue: int, expenses: int) -> str:
    net_income = revenue - expenses
    report  = f"Financial Report for {company_name}\n"
    report += f"Revenue: ${revenue}\n"
    report += f"Expenses: ${expenses}\n"
    report += f"Net Income: ${net_income}\n"
    return report

# examine the function: .name, .description, .args, .return_type

Integrating the custom tools

In [ ]:
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(
    model="",
    api_key=""
)

agent = create_react_agent(llm=llm, tools=[financial_report])

messages = agent.invoke({"messages" : [{"human": "Generate a financial report for ABC Corp with revenue of 100000 and expenses of 75000."}]})

## Integrating Document Loaders

![alt text](<Screenshot 2026-10-03 114754.png>)

- Use embeddings to retrieve relevant information to integrate into the prompt

![alt text](<Screenshot 2026-10-03 115042.png>)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Load a PDF file and extract its content
pdf_loader = PyPDFLoader("path/to/pdf/file.pdf")
pdf_data = pdf_loader.load()
print(pdf_data[0].page_content)

csv_loader = CSVLoader("path/to/csv/file.csv")
csv_data = csv_loader.load()
print(csv_data[0].page_content)

html_loader = UnstructuredHTMLLoader("path/to/html/file.html")
html_data = html_loader.load()
print(html_data[0].page_content)

### Splitting External Data for Retrieval

![alt text](<Screenshot 2026-10-03 120331.png>)

- Chunk Overlap to retain the information
- Document Splitting Strategy:
    - CharacterTextSplitter
    - RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitter import CharacterTextSplitter

ct_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200
)

docs = ct_splitter.split_documents(pdf_data)

In [ ]:
from langchain_text_splitter import ReCharacterTextSplitter

rct_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=200
)

docs = rct_splitter.split_documents(pdf_data)

### RAG Storage & Retrieval Using Vector Databases

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_function = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=""
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_function,
    persist_directory="path/to/persist/directory"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
# prompt template for the retrieval-augmented generation (RAG) chain

from langchain_core.prompts import ChatPromptTemplate

message ="""
Review the following context and answer the question below. If the context does not contain enough information to answer the question, say "I don't know".

Guidelines:
{guidelines}

Copy:
{copy}

Fixed Copy:
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("human", message)
])


### Chaining All Together

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = ({"guidelines": retriever, "copy": RunnablePassthrough()}
             | prompt_template
             | llm  
             )

response = rag_chain.invoke("What is the capital of France?")